# qdmpy_core Tutorial: ODMR Data Processing and Analysis

This tutorial provides a comprehensive introduction to the qdmpy_core package for processing and analyzing Optically Detected Magnetic Resonance (ODMR) data in Quantum Diamond Microscopy (QDM).

## Contents
1. [Overview of qdmpy_core](#1.-Overview-of-qdmpy_core)
2. [Installation and Setup](#2.-Installation-and-Setup)
3. [Loading ODMR Data](#3.-Loading-ODMR-Data)
4. [Processing ODMR Data](#4.-Processing-ODMR-Data)
5. [Automatic Model Selection](#5.-Automatic-Model-Selection)
6. [Fitting ODMR Data](#6.-Fitting-ODMR-Data)
7. [Creating Measurements](#7.-Creating-Measurements)
8. [Visualizing Results](#8.-Visualizing-Results)
9. [Working with Real Data](#9.-Working-with-Real-Data)
10. [Advanced Usage](#10.-Advanced-Usage)

# QDMpy Tutorial: ODMR Data Processing and Analysis

This tutorial provides a comprehensive introduction to the QDMpy package for processing and analyzing Optically Detected Magnetic Resonance (ODMR) data in Quantum Diamond Microscopy (QDM).

## Contents
1. [Overview of QDMpy](#1.-Overview-of-QDMpy)
2. [Installation and Setup](#2.-Installation-and-Setup)
3. [Loading ODMR Data](#3.-Loading-ODMR-Data)
4. [Processing ODMR Data](#4.-Processing-ODMR-Data)
5. [Automatic Model Selection](#5.-Automatic-Model-Selection)
6. [Fitting ODMR Data](#6.-Fitting-ODMR-Data)
7. [Creating Measurements](#7.-Creating-Measurements)
8. [Working with Image Data](#8.-Working-with-Image-Data)
9. [Visualizing Results](#9.-Visualizing-Results)
10. [Working with Real Data](#10.-Working-with-Real-Data)
11. [Advanced Usage](#11.-Advanced-Usage)
    - [Custom Processing Pipelines](#11.1-Custom-Processing-Pipelines)
    - [Custom Models](#11.2-Custom-Models)
    - [Working with ROIs](#11.3-Working-with-Specific-Regions-of-Interest-(ROIs))
    - [Using the IO Module](#11.4-Using-the-IO-Module)
    - [Package Structure](#11.5-Package-Structure-and-Organization)

## 2. Installation and Setup

qdmpy_core requires Python 3.12 or higher. It can be installed using pip or uv:

In [ ]:
# If you haven't installed qdmpy_core yet, uncomment and run the line below
# !pip install qdmpy

# For development installation (if you're working with the source code)
# !uv pip install -e .

In [1]:
import sys
sys.path.append("~/git/qdmpy_core/src")

In [ ]:
# Import main QDMpy modules
import os
import numpy as np
import matplotlib.pyplot as plt

# Set figure size for better visualization
plt.rcParams['figure.figsize'] = (12, 8)

# Import QDMpy modules
import QDMpy
from QDMpy.odmr import ODMRData, ODMR
from QDMpy.odmr.io import MatlabLoader
from QDMpy.odmr.processors import (
    ODMRProcessorManager, 
    BinningProcessor, 
    NormalizationProcessor
)
from QDMpy.models import ModelRegistry, ESR14N, ESR15N, ESRSINGLE
from QDMpy.guess import (
    guess_model, 
    guess_n_peaks, 
    guess_initial_fit_parameters
)
from QDMpy.measurement import Measurement
from QDMpy.io import has_csv, get_image, get_image_file

# Check QDMpy version
print(f"QDMpy version: {QDMpy.__version__ if hasattr(QDMpy, '__version__') else 'Development version'}")

## 3. Loading ODMR Data

qdmpy_core can load ODMR data from various file formats. In this tutorial, we'll focus on loading data from MATLAB (.mat) files, which is a common format for experimental data.

### 3.1 Understanding ODMR Data Structure

Before loading data, it's important to understand the structure of ODMR data:

- **4D Data Structure**: ODMR data typically has dimensions (n_polarity, n_freq_range, n_frequencies, n_pixels)
  - n_polarity: Number of polarities (usually 2 corresponding to positive and negative B fields)
  - n_freq_range: Number of frequency ranges scanned
  - n_frequencies: Number of frequency points in each range
  - n_pixels: Number of pixels in the spatial dimension
  
- **Scan Dimensions**: Spatial dimensions of the scan (height, width)
  
- **Frequency Array**: The frequency values corresponding to the data points

### 3.2 Loading from MATLAB Files

In [ ]:
# Define the data folder where test files are located
data_folder = "./tests/data"

# Initialize the MatlabLoader
loader = MatlabLoader(data_folder=data_folder)

# Load the ODMR data
raw_data, scan_dimensions, frequencies = loader.load()

# Print information about the loaded data
print(f"Raw data shape: {raw_data.shape}")
print(f"Scan dimensions: {scan_dimensions}")
print(f"Frequency range: {frequencies.min()/1e9:.4f} GHz to {frequencies.max()/1e9:.4f} GHz")
print(f"Number of frequency points: {len(frequencies)}")

### 3.3 Creating an ODMRData Object

Once the raw data is loaded, we can create an ODMRData object, which is the fundamental container for ODMR data in qdmpy_core.

In [ ]:
# Create an ODMRData object from the loaded data
odmr_data = ODMRData(
    data=raw_data,
    scan_dimensions=scan_dimensions,
    frequencies=frequencies
)

# Print information about the ODMRData object
print(f"ODMRData shape: {odmr_data.shape}")
print(f"ODMRData metadata: {odmr_data.metadata}")

### 3.4 Exploring the Raw Data

Let's visualize some of the raw ODMR data to get a better understanding.

In [ ]:
# Helper function to plot ODMR spectra
def plot_odmr_spectrum(data, frequencies, pixel_index=0, title="ODMR Spectrum"):
    plt.figure(figsize=(10, 6))
    
    # Assuming first polarity and frequency range
    polarity_idx = 0
    freq_range_idx = 0
    
    # Frequencies are in Hz, convert to GHz for better readability
    freq_ghz = frequencies / 1e9
    
    # Extract spectrum for the selected pixel
    spectrum = data[polarity_idx, freq_range_idx, :, pixel_idx]
    
    plt.plot(freq_ghz, spectrum, 'b-', linewidth=2)
    plt.xlabel('Frequency (GHz)')
    plt.ylabel('Intensity (a.u.)')
    plt.title(f"{title} - Pixel {pixel_idx}")
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()

# Plot ODMR spectrum for a sample pixel
plot_odmr_spectrum(odmr_data.data, odmr_data.frequencies, pixel_index=10, title="Raw ODMR Spectrum")

## 4. Processing ODMR Data

Raw ODMR data often needs processing to improve signal quality, reduce noise, and prepare for fitting. qdmpy_core provides a flexible processor system that allows for various data transformations.

### 4.1 Creating an ODMR Manager

First, we create an ODMR manager that will handle the data processing.

In [ ]:
# Create an ODMR manager
odmr = ODMR(odmr_data)

# Print information about the ODMR manager
print(f"ODMR data shape: {odmr.raw_data.shape}")
print(f"ODMR processor chain: {odmr.processor_manager.processors if odmr.processor_manager.processors else 'Empty'}")

### 4.2 Adding Processors

qdmpy_core includes several built-in processors for common operations. Let's add some processors to our ODMR manager:

In [ ]:
# Initialize processor manager (if not using the one already in ODMR object)
# processor_manager = ODMRProcessorManager()

# Add a binning processor (reduces data size by binning neighboring pixels)
odmr.processor_manager.add_processor(BinningProcessor(bin_factor=2))

# Add a normalization processor (optional - normalizes data to make comparisons easier)
odmr.processor_manager.add_processor(NormalizationProcessor(method="max"))

# Print the processor chain
print("ODMR processor chain:")
for i, processor in enumerate(odmr.processor_manager.processors):
    print(f"  {i+1}. {processor.__class__.__name__} - {processor}")

### 4.3 Processing the Data

Now, let's process the data using the configured processors.

In [ ]:
# Process the data
odmr.process_data()

# Print information about the processed data
print(f"Processed data shape: {odmr.processed_data.shape}")

# Compare raw and processed data shapes
print(f"Raw data shape: {odmr.raw_data.shape}")
print(f"Processed data shape: {odmr.processed_data.shape}")

# Plot the processed ODMR spectrum for the same pixel as before
# Note that the pixel index might need adjustment due to binning
processed_pixel_idx = 5  # Adjust based on binning factor
plot_odmr_spectrum(odmr.processed_data.data, odmr.processed_data.frequencies, 
                   pixel_index=processed_pixel_idx, title="Processed ODMR Spectrum")

## 5. Automatic Model Selection

One of the powerful features of qdmpy_core is its ability to automatically select an appropriate model based on the number of resonance peaks in the ODMR spectrum. This is handled by the `guess_model` and related functions in the `guess` module.

### 5.1 Detecting the Number of Peaks

In [ ]:
# Detect the number of peaks in the ODMR data
n_peaks, doubt, peak_indices = guess_n_peaks(odmr.processed_data.data)

print(f"Detected {n_peaks} peaks in the ODMR data")
print(f"Confidence level: {'Low (uncertain)' if doubt else 'High (certain)'}")

### 5.2 Selecting a Model Based on Peak Count

qdmpy_core includes several predefined models for ODMR spectra, each suited to a specific number of resonance peaks:

In [ ]:
# List all available models
print("Available models in qdmpy_core:")
for name, model_info in ModelRegistry.all().items():
    model_instance = model_info["class"]()
    print(f"  - {name}: {model_instance.n_peaks} peaks, {len(model_instance.parameters_unique)} parameters")
    print(f"    Parameters: {model_instance.parameters_unique}")

# Get the model based on the detected number of peaks
try:
    model = get_model_by_peaks(n_peaks)
    print(f"\nSelected model: {model.name} with {model.n_peaks} peaks")
    print(f"Model parameters: {model.parameters_unique}")
except ValueError as e:
    print(f"\nError selecting model: {e}")
    # Fallback to a default model if needed
    model = ESRSINGLE()
    print(f"Falling back to default model: {model.name}")

### 5.3 Guessing Initial Parameters

Once we have a model, we need to estimate initial parameters for the fitting procedure. qdmpy_core provides functions to guess these parameters based on the data.

In [ ]:
# Guess initial fit parameters
initial_parameters = guess_initial_fit_parameters(
    data=odmr.processed_data.data,
    freq=odmr.processed_data.frequencies,
    model=model
)

print(f"Initial parameters shape: {initial_parameters.shape}")

# Display initial parameters for a sample pixel
sample_pixel = 0
polarity_idx = 0
freq_range_idx = 0

print(f"\nInitial parameters for pixel {sample_pixel}:")
for i, param in enumerate(model.parameters_unique):
    value = initial_parameters[polarity_idx, freq_range_idx, sample_pixel, i]
    if 'center' in param:
        # Convert Hz to GHz for better readability
        print(f"  {param}: {value/1e9:.6f} GHz")
    else:
        print(f"  {param}: {value:.6f}")

## 6. Fitting ODMR Data

With a selected model and initial parameters, we can now fit the ODMR data. qdmpy_core leverages GPU acceleration (when available) through the Gpufit library to perform fast fitting across all pixels.

### 6.1 Creating a Fit Object

First, let's create a fit object to handle the fitting process.

In [ ]:
# Import the Fit class
from qdmpy_core.fit import Fit

# Create a fit object
fit_obj = Fit(
    data=odmr.processed_data.data,
    frequencies=odmr.processed_data.frequencies,
    model_name=model.name
)

print(f"Fit object created with model: {fit_obj.model_name}")
print(f"Fit parameters: {fit_obj.model_params_unique}")

### 6.2 Setting Fit Constraints

We can set constraints on the fit parameters to improve the fitting process.

In [ ]:
# Set constraints for the fit parameters
# Default constraints are already set based on the configuration

# Get the default constraints
print("Default constraints:")
for param, constraint in fit_obj.constraints.items():
    min_val, max_val, constraint_type, unit = constraint
    print(f"  {param}: [{min_val}, {max_val}] ({constraint_type}, {unit})")

# Optionally, set custom constraints
# For example, setting tighter constraints on the center frequency
if 'center' in fit_obj.model_params_unique:
    center_freq = initial_parameters[0, 0, 0, fit_obj.model_params_unique.index('center')]
    fit_obj.set_constraints(
        'center',
        center_freq - 10e6,  # 10 MHz below guessed center
        center_freq + 10e6,  # 10 MHz above guessed center
        'BOUND'  # Use bounded constraint type
    )

### 6.3 Performing the Fit

In [ ]:
# Perform the fit
fit_obj.fit_odmr()

# Check if the fit was successful
print(f"Fit completed. Success: {fit_obj.fitted}")

# Get the fit results
fit_results = fit_obj.parameter
print(f"Fit results shape: {fit_results.shape}")

### 6.4 Visualizing the Fit Results

Let's visualize the fit results for a sample pixel.

In [ ]:
# Function to plot the fit results for a specific pixel
def plot_fit_results(fit_obj, pixel_idx=0, polarity_idx=0, freq_range_idx=0):
    # Get the data for the selected pixel
    data = fit_obj._data[polarity_idx, freq_range_idx, :, pixel_idx]
    frequencies = fit_obj.frequencies
    freq_ghz = frequencies / 1e9  # Convert to GHz for better readability
    
    # Get the fit parameters for this pixel
    params = fit_obj.parameter[polarity_idx, freq_range_idx, pixel_idx]
    
    # Generate the fitted curve using the model and fit parameters
    fitted_curve = fit_obj.model_func(frequencies, params)
    
    # Create the plot
    plt.figure(figsize=(12, 8))
    plt.plot(freq_ghz, data, 'bo', label='Data')
    plt.plot(freq_ghz, fitted_curve, 'r-', label='Fit', linewidth=2)
    
    # Add annotations for fit parameters
    param_text = ""
    for i, param in enumerate(fit_obj.model_params_unique):
        value = params[i]
        if 'center' in param:
            param_text += f"{param}: {value/1e9:.6f} GHz\n"
        else:
            param_text += f"{param}: {value:.6f}\n"
    
    # Add the parameter text to the plot
    plt.annotate(param_text, xy=(0.02, 0.02), xycoords='axes fraction',
                 bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="gray", alpha=0.8))
    
    plt.xlabel('Frequency (GHz)')
    plt.ylabel('Intensity (a.u.)')
    plt.title(f'ODMR Fit Results - Pixel {pixel_idx}')
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()

# Plot fit results for a sample pixel
sample_pixel = 5
plot_fit_results(fit_obj, pixel_idx=sample_pixel)

## 7. Creating Measurements

qdmpy_core provides a Measurement class that combines ODMR data with associated imagery (e.g., light and laser images) to create a comprehensive representation of a QDM experiment.

### 7.1 Loading Sample Images

In [ ]:
# Load sample light and laser images
# In a real experiment, you would load these from files
# For this tutorial, we'll create dummy images that match the processed data dimensions

# Extract the spatial dimensions from the processed data
spatial_dims = odmr.processed_data.scan_dimensions
height, width = spatial_dims

# Create dummy light and laser images
light_image = np.random.random((height, width))
laser_image = np.random.random((height, width))

# Try to load real images if available using QDMpy's IO module
try:
    # Check if CSV files are available in the data folder
    file_list = os.listdir(data_folder)
    
    # Use QDMpy's io functions to load images
    if has_csv(file_list):
        print("CSV files detected in data folder.")
        
        # Try to load light image
        try:
            light_image = get_image(data_folder, [f for f in file_list if "led" in f.lower()])
            print("Successfully loaded light image using QDMpy.io")
        except ValueError as e:
            print(f"Could not load light image: {e}")
        
        # Try to load laser image
        try:
            laser_image = get_image(data_folder, [f for f in file_list if "laser" in f.lower()])
            print("Successfully loaded laser image using QDMpy.io")
        except ValueError as e:
            print(f"Could not load laser image: {e}")
    else:
        print("No CSV files found in data folder, using dummy images.")
except Exception as e:
    print(f"Error loading images: {e}")
    print("Using dummy images instead.")

# Plot the light and laser images
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].imshow(light_image, cmap='viridis')
axes[0].set_title('Light Image')
axes[0].axis('off')

axes[1].imshow(laser_image, cmap='inferno')
axes[1].set_title('Laser Image')
axes[1].axis('off')

plt.tight_layout()
plt.show()

### 7.2 Creating a Measurement Object

In [ ]:
# Create a measurement object
measurement = Measurement(
    odmr=odmr,
    light_image=light_image,
    laser_image=laser_image,
    output_directory="./output",  # Directory to save results
    pixel_spacing=4e-6,  # Pixel spacing in meters (4 μm)
    fit_model=model.name  # Use the model we determined earlier
)

# Print information about the measurement
print(measurement)

### 7.3 Adding Metadata

We can add metadata to the measurement object to keep track of experimental details.

In [ ]:
# Add metadata to the measurement
measurement.metadata.update({
    "experiment_date": "2025-03-09",
    "sample": "Test sample",
    "temperature": 298.15,  # K
    "magnetic_field": 5e-3,  # T
    "laser_power": 100,  # mW
    "notes": "Tutorial measurement"
})

# Print the updated metadata
print("Measurement metadata:")
for key, value in measurement.metadata.items():
    print(f"  {key}: {value}")

## 8. Visualizing Results

qdmpy_core provides various visualization tools to explore ODMR data and fitting results.

### 8.1 Plotting Fit Parameters

In [ ]:
# Import qdmpy_core plotting module
from qdmpy_core.plotting import plot_parameter_maps

# Check if fit parameters are available, if not, perform the fit
if not hasattr(measurement, '_fit') or not measurement._fit.fitted:
    print("Fitting the ODMR data...")
    measurement._fit = fit_obj

# Plot parameter maps (e.g., center frequency, contrast, width)
for param_name in measurement._fit.model_params_unique:
    try:
        # Get parameter index
        param_idx = measurement._fit.model_params_unique.index(param_name)
        
        # Extract parameter values (first polarity and frequency range)
        param_values = measurement._fit.parameter[0, 0, :, param_idx]
        
        # Reshape to match the spatial dimensions
        height, width = measurement.odmr.processed_data.scan_dimensions
        param_map = param_values.reshape(height, width)
        
        # Create the plot
        plt.figure(figsize=(10, 8))
        
        # Handle different parameter types appropriately
        if 'center' in param_name:
            # Convert Hz to GHz for center frequencies
            plt.imshow(param_map / 1e9, cmap='plasma')
            plt.colorbar(label=f"{param_name} (GHz)")
        else:
            plt.imshow(param_map, cmap='viridis')
            plt.colorbar(label=param_name)
        
        plt.title(f"{param_name.capitalize()} Map")
        plt.axis('off')
        plt.tight_layout()
        plt.show()
    except Exception as e:
        print(f"Error plotting {param_name}: {e}")

### 8.2 Custom Visualizations

We can create custom visualizations to highlight specific aspects of the data.

In [ ]:
# Create a combined visualization of light image and center frequency
try:
    # Get center frequency parameter
    center_idx = measurement._fit.model_params_unique.index('center')
    center_values = measurement._fit.parameter[0, 0, :, center_idx]
    
    # Reshape to match spatial dimensions
    height, width = measurement.odmr.processed_data.scan_dimensions
    center_map = center_values.reshape(height, width)
    
    # Create a combined visualization
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    
    # Light image
    im1 = axes[0].imshow(measurement.light_image, cmap='gray')
    axes[0].set_title('Light Image')
    axes[0].axis('off')
    plt.colorbar(im1, ax=axes[0], shrink=0.8)
    
    # Center frequency map
    im2 = axes[1].imshow(center_map / 1e9, cmap='plasma')  # Convert to GHz
    axes[1].set_title('Center Frequency (GHz)')
    axes[1].axis('off')
    plt.colorbar(im2, ax=axes[1], shrink=0.8)
    
    # Overlay (center frequency with light image contours)
    im3 = axes[2].imshow(center_map / 1e9, cmap='plasma', alpha=0.8)
    contour = axes[2].contour(measurement.light_image, colors='white', alpha=0.5, levels=5)
    axes[2].set_title('Overlay: Center Frequency with Light Contours')
    axes[2].axis('off')
    plt.colorbar(im3, ax=axes[2], shrink=0.8)
    
    plt.tight_layout()
    plt.show()
except Exception as e:
    print(f"Error creating combined visualization: {e}")

## 9. Working with Real Data

Let's apply what we've learned to work with real data from start to finish.

### 9.1 Complete Workflow with Real Data

In [ ]:
# Define a complete workflow function
def process_odmr_data(data_folder, output_folder="./output", bin_factor=2):
    """Process ODMR data from a folder and save the results."""
    print(f"Processing ODMR data from {data_folder}...")
    
    # Step 1: Load the data
    print("\nStep 1: Loading data...")
    loader = MatlabLoader(data_folder=data_folder)
    raw_data, scan_dimensions, frequencies = loader.load()
    print(f"Loaded data with shape {raw_data.shape} and {len(frequencies)} frequency points")
    
    # Step 2: Create ODMR object and process the data
    print("\nStep 2: Processing data...")
    odmr_data = ODMRData(data=raw_data, scan_dimensions=scan_dimensions, frequencies=frequencies)
    odmr = ODMR(odmr_data)
    
    # Add processors
    odmr.processor_manager.add_processor(BinningProcessor(bin_factor=bin_factor))
    odmr.processor_manager.add_processor(NormalizationProcessor(method="max"))
    
    # Process the data
    odmr.process_data()
    print(f"Processed data shape: {odmr.processed_data.shape}")
    
    # Step 3: Automatic model selection
    print("\nStep 3: Selecting model...")
    try:
        n_peaks, doubt, _ = guess_n_peaks(odmr.processed_data.data)
        print(f"Detected {n_peaks} peaks with {'low' if doubt else 'high'} confidence")
        
        # Get model based on peak count
        model = guess_model(n_peaks)
        print(f"Selected model: {model.name} with {model.n_peaks} peaks")
    except Exception as e:
        print(f"Error in model selection: {e}")
        # Fall back to ESRSINGLE
        model = ModelRegistry.get("ESRSINGLE")
        print(f"Falling back to default model: {model.name}")
    
    # Step 4: Fit the data
    print("\nStep 4: Fitting ODMR data...")
    fit_obj = Fit(
        data=odmr.processed_data.data,
        frequencies=odmr.processed_data.frequencies,
        model_name=model.name
    )
    fit_obj.fit_odmr()
    print(f"Fit completed. Success: {fit_obj.fitted}")
    
    # Step 5: Create measurement object
    print("\nStep 5: Creating measurement object...")
    
    # Load images using QDMpy's io module
    height, width = odmr.processed_data.scan_dimensions
    light_image = np.random.random((height, width))  # Default dummy image
    laser_image = np.random.random((height, width))  # Default dummy image
    
    try:
        # Get list of files in data folder
        file_list = os.listdir(data_folder)
        
        # Check if we have CSV files
        if QDMpy.io.has_csv(file_list):
            # Try to load light image
            try:
                light_candidates = [f for f in file_list if "led" in f.lower()]
                if light_candidates:
                    light_image = QDMpy.io.get_image(data_folder, light_candidates)
                    if light_image.shape != (height, width):
                        print(f"Light image shape {light_image.shape} doesn't match required {(height, width)}")
                        light_image = np.random.random((height, width))
                    else:
                        print("Successfully loaded light image")
            except ValueError as e:
                print(f"Could not load light image: {e}")
            
            # Try to load laser image
            try:
                laser_candidates = [f for f in file_list if "laser" in f.lower()]
                if laser_candidates:
                    laser_image = QDMpy.io.get_image(data_folder, laser_candidates)
                    if laser_image.shape != (height, width):
                        print(f"Laser image shape {laser_image.shape} doesn't match required {(height, width)}")
                        laser_image = np.random.random((height, width))
                    else:
                        print("Successfully loaded laser image")
            except ValueError as e:
                print(f"Could not load laser image: {e}")
        else:
            print("No CSV files found in data folder, using dummy images")
    except Exception as e:
        print(f"Error loading images: {e}")
        print("Using dummy images instead")
    
    # Create the measurement object
    measurement = Measurement(
        odmr=odmr,
        light_image=light_image,
        laser_image=laser_image,
        output_directory=output_folder,
        pixel_spacing=4e-6,  # 4 μm
        fit_model=model.name
    )
    
    # Add metadata
    measurement.metadata.update({
        "experiment_date": "2025-03-09",
        "sample": "Real data sample",
        "processing": f"Binning factor: {bin_factor}, Normalization: max"
    })
    
    # Set the fit object
    measurement._fit = fit_obj
    
    print("\nProcessing completed successfully!")
    return measurement

# Process the sample data
real_measurement = process_odmr_data(data_folder)

### 9.2 Visualizing the Results

In [ ]:
# Create visualization of the results
if real_measurement and real_measurement._fit and real_measurement._fit.fitted:
    # Visualize fit parameters
    try:
        # Get center frequency parameter
        center_idx = real_measurement._fit.model_params_unique.index('center')
        center_values = real_measurement._fit.parameter[0, 0, :, center_idx]
        
        # Reshape to match spatial dimensions
        height, width = real_measurement.odmr.processed_data.scan_dimensions
        center_map = center_values.reshape(height, width)
        
        # Get contrast parameter if available
        contrast_map = None
        try:
            contrast_idx = [i for i, p in enumerate(real_measurement._fit.model_params_unique) if 'contrast' in p][0]
            contrast_values = real_measurement._fit.parameter[0, 0, :, contrast_idx]
            contrast_map = contrast_values.reshape(height, width)
        except (IndexError, ValueError):
            print("Could not extract contrast parameter")
        
        # Create visualization
        if contrast_map is not None:
            fig, axes = plt.subplots(2, 2, figsize=(12, 10))
            
            # Light image
            im1 = axes[0, 0].imshow(real_measurement.light_image, cmap='gray')
            axes[0, 0].set_title('Light Image')
            axes[0, 0].axis('off')
            plt.colorbar(im1, ax=axes[0, 0])
            
            # Laser image
            im2 = axes[0, 1].imshow(real_measurement.laser_image, cmap='hot')
            axes[0, 1].set_title('Laser Image')
            axes[0, 1].axis('off')
            plt.colorbar(im2, ax=axes[0, 1])
            
            # Center frequency map
            im3 = axes[1, 0].imshow(center_map / 1e9, cmap='plasma')  # Convert to GHz
            axes[1, 0].set_title('Center Frequency (GHz)')
            axes[1, 0].axis('off')
            plt.colorbar(im3, ax=axes[1, 0])
            
            # Contrast map
            im4 = axes[1, 1].imshow(contrast_map, cmap='viridis')
            axes[1, 1].set_title('Contrast')
            axes[1, 1].axis('off')
            plt.colorbar(im4, ax=axes[1, 1])
        else:
            # Simplified visualization without contrast
            fig, axes = plt.subplots(1, 3, figsize=(18, 6))
            
            # Light image
            im1 = axes[0].imshow(real_measurement.light_image, cmap='gray')
            axes[0].set_title('Light Image')
            axes[0].axis('off')
            plt.colorbar(im1, ax=axes[0])
            
            # Laser image
            im2 = axes[1].imshow(real_measurement.laser_image, cmap='hot')
            axes[1].set_title('Laser Image')
            axes[1].axis('off')
            plt.colorbar(im2, ax=axes[1])
            
            # Center frequency map
            im3 = axes[2].imshow(center_map / 1e9, cmap='plasma')  # Convert to GHz
            axes[2].set_title('Center Frequency (GHz)')
            axes[2].axis('off')
            plt.colorbar(im3, ax=axes[2])
        
        plt.tight_layout()
        plt.show()
        
        # Plot a sample spectrum with fit
        sample_pixel = height * width // 2  # Middle pixel
        plot_fit_results(real_measurement._fit, pixel_idx=sample_pixel)
    except Exception as e:
        print(f"Error creating visualization: {e}")
else:
    print("No fit results available for visualization.")

## 10. Advanced Usage

QDMpy provides several advanced features for more sophisticated data analysis. Here's a brief overview of some advanced usage patterns.

In [ ]:
### 10.1 Custom Processing Pipelines

You can create custom processors by extending the `ODMRProcessor` base class.

### 10.2 Custom Models

You can create custom models by extending the `Model` base class and registering them with the `ModelRegistry`.

In [ ]:
### 10.2 Custom Models

You can create custom models by extending the `Model` base class and registering them with the `ModelRegistry`.

### 10.3 Working with Specific Regions of Interest (ROIs)

In [ ]:
### 10.3 Working with Specific Regions of Interest (ROIs)

In [ ]:
### 10.4 Using the IO Module

QDMpy includes a dedicated io module for handling image loading and other file operations. This module provides functions for finding and loading image files (CSV, JPG, etc.) in a consistent and error-handled way.